In [15]:
# import libraries
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.preprocessing.image import load_img
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np
import os

from sklearn.metrics import confusion_matrix
# initialize the initial learning rate, number of epochs to train for and batch size
INIT_LR = 1e-4
EPOCHS = 20 
BS = 20

DIRECTORY=r"D:\AI course\AI\Deep Learning\Dog_Behavior_Detection\Dataset" 
CATEGORIES=["Aggressive","Normal"]


# load and preprocess the images
print("[INFO] loading images...")

data = []
labels = []

for category in CATEGORIES:
    path = os.path.join(DIRECTORY, category)
    for img in os.listdir(path):
    	img_path = os.path.join(path, img)
    	image = load_img(img_path, target_size=(224, 224))
    	image = img_to_array(image)
    	image = preprocess_input(image)

    	data.append(image)
    	labels.append(category)

# perform one-hot encoding on the labels
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = labels.flatten() 


data = np.array(data, dtype="float32")
labels = np.array(labels)
print(lb.classes_)

(trainX, testX, trainY, testY) = train_test_split(data, labels,
	test_size=0.20, stratify=labels, random_state=42)


[INFO] loading images...


C:\Anaconda3\envs\facemask\lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


['Aggressive' 'Normal']


In [17]:
# construct the training image generator for data augmentation
aug = ImageDataGenerator(
	rotation_range=20,
	zoom_range=0.15,
	width_shift_range=0.2,
	height_shift_range=0.2,
	shear_range=0.15,
	horizontal_flip=True,
	fill_mode="nearest")

In [23]:
# load the EfficientNetB0 network, ensuring the head FC layer sets are left off
baseModel = EfficientNetB0(weights="imagenet", include_top=False,
	input_tensor=Input(shape=(224, 224, 3)))

# construct the head of the model that will be placed on top of the base model
headModel = baseModel.output 
headModel = MaxPooling2D(pool_size=(2,2))(headModel)
headModel = Flatten()(headModel)
headModel = Dense(128, activation="relu")(headModel)
headModel = Dropout(0.5)(headModel)
headModel = Dense(1, activation="sigmoid")(headModel)


# place the head FC model on top of the base model (this will become the actual model we will train)
model = Model(inputs=baseModel.input, outputs=headModel)

# loop over all layers in the base model and freeze them so they will *not* be updated during the first training process
for layer in baseModel.layers:
	layer.trainable = False
    
# compile our model
print("[INFO] compiling model...")
opt = Adam(learning_rate=INIT_LR)
model.compile(loss="binary_crossentropy", optimizer=opt,
	metrics=["accuracy"])

# train the head of the network
print("[INFO] training head...")
H = model.fit(
	aug.flow(trainX, trainY, batch_size=BS),
	validation_data=(testX, testY),
	epochs=EPOCHS)
# make predictions on the testing set
print("[INFO] evaluating network...")
predIdxs = model.predict(testX, batch_size=BS)

# convert predicted probabilities into binary class labels using a 0.5 threshold
predIdxs = (predIdxs > 0.5).astype("int32")
predIdxs = predIdxs.flatten()


print(classification_report(testY, predIdxs, target_names=CATEGORIES))
print(f"confusion_matrix\n{confusion_matrix(testY,predIdxs)}")

# serialize the model to disk
print("[INFO] saving dog_behavior detector model...")
import tensorflow as tf
tf.saved_model.save(model, "dog_behavior_detector_eff_tfg")

[INFO] compiling model...
[INFO] training head...
Epoch 1/20
56/56 [==============================] - 29s 458ms/step - loss: 0.9067 - accuracy: 0.6446 - val_loss: 0.4285 - val_accuracy: 0.8000
Epoch 2/20
56/56 [==============================] - 25s 441ms/step - loss: 0.5682 - accuracy: 0.7339 - val_loss: 0.3712 - val_accuracy: 0.8286
Epoch 3/20
56/56 [==============================] - 25s 440ms/step - loss: 0.4650 - accuracy: 0.7768 - val_loss: 0.3760 - val_accuracy: 0.8214
Epoch 4/20
56/56 [==============================] - 25s 451ms/step - loss: 0.4572 - accuracy: 0.7893 - val_loss: 0.3471 - val_accuracy: 0.8500
Epoch 5/20
56/56 [==============================] - 25s 441ms/step - loss: 0.3740 - accuracy: 0.8366 - val_loss: 0.3038 - val_accuracy: 0.8714
Epoch 6/20
56/56 [==============================] - 25s 438ms/step - loss: 0.3701 - accuracy: 0.8259 - val_loss: 0.3197 - val_accuracy: 0.8464
Epoch 7/20
56/56 [==============================] - 25s 438ms/step - loss: 0.3473 - accuracy

INFO:tensorflow:Assets written to: dog_behavior_detector_eff_tfg\assets


INFO:tensorflow:Assets written to: dog_behavior_detector_eff_tfg\assets
